# VaR

## Data

This problem uses `weekly` return data from `data/spx_returns_weekly.xlsx`.

Choose any `4` stocks to evaluate below.

For example, 
* `AAPL`
* `META`
* `NVDA`
* `TSLA`

# 1 Diversification

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILEDATA = 'spx_returns_weekly.xlsx'
TICKS = ["AAPL", "META", "NVDA", "TSLA"]

In [31]:
df = pd.read_excel(FILEDATA, sheet_name="spx returns")
cols = ["date"] + TICKS
data = df[cols]
data = df[cols].copy()

In [33]:
#1.1
vol=data[TICKS].std()
print(vol)

var=data[TICKS].quantile(0.05)
print(var)

cvar= {}
for t in TICKS:
    threshold = var[t]
    cvar[t] = data[t][data[t] <= threshold].mean()

cvar= pd.Series(cvar)
print(cvar)


AAPL    0.038585
META    0.049541
NVDA    0.062905
TSLA    0.080262
dtype: float64
AAPL   -0.054619
META   -0.072309
NVDA   -0.085785
TSLA   -0.116946
Name: 0.05, dtype: float64
AAPL   -0.082999
META   -0.106430
NVDA   -0.115317
TSLA   -0.147824
dtype: float64


In [43]:
#1.2
data['portfolio']=data[TICKS].mean(axis=1)
print(data[['date', 'portfolio']].head())

vol=data['portfolio'].std()
print(vol)

var=data['portfolio'].quantile(0.05)
print(var)

cvar= data['portfolio'][data['portfolio'] <= var].mean()
print(cvar)

# We notice that the volatility of the portfolio is lower than all the individual ticker volatilities except for AAPL.  
# We can see that the portfolio benefitted from diversification because even though each stock had risk and volatility,
# when we combined them into an evenly weighted portfolio the variance was lower than what an average of the variance of individual stocks would give.
# What drives this result mathematically is the way that the portfolio combines the data BEFORE squaring for standard deviation calculations.
# This means that when one stock has a higher variance but the others do not it can be made small whereas variance can only get high when they all move together.  



        date  portfolio
0 2015-07-03   0.011063
1 2015-07-10  -0.031132
2 2015-07-17   0.051806
3 2015-07-24  -0.021173
4 2015-07-31  -0.006492
0.04322864533764943
-0.060572519860502834
-0.08417548345387538


In [51]:
#1.3
data['portfolio_no_TSLA']=(data['AAPL'] + data['NVDA'] + data['META'] + 0)/4


vol_no_TSLA = data['portfolio_no_TSLA'].std()
var_no_TSLA = data['portfolio_no_TSLA'].quantile(0.05)
cvar_no_TSLA = data['portfolio_no_TSLA'][data['portfolio_no_TSLA'] <= var_no_TSLA].mean()

print("Volatility (no TSLA):", vol_no_TSLA)
print("VaR(.05) (no TSLA):", var_no_TSLA)
print("CVaR(.05) (no TSLA):", cvar_no_TSLA)

# We see that TSLA is adding about 1.3% of volatility to the portfolio.  However, alone it had 8% and weighted at 1/4
# we would expect around 2% of volatility to the portfolio but we see this isn't the case.
# This is because some of the change could be happening when others move with it.

Volatility (no TSLA): 0.030057997997142028
VaR(.05) (no TSLA): -0.04222057190480661
CVaR(.05) (no TSLA): -0.06000072040867311


## 1.1

Using the full sample, calculate for each series the (unconditional) 
* volatility
* empirical VaR (.05)
* empirical CVaR (.05)

Recall that by **empirical** we refer to the direct quantile estimation. (For example, using `.quantile()` in pandas.

## 1.2
Form an equally-weighted portfolio of the investments.

Calculate the statistics of `1.1` for this portfolio, and compare the results to the individual return statistics. What do you find? What is driving this result?

## 1.3
Re-calculate `1.2`, but this time drop your most volatile asset, and replace the portion it was getting with 0. (You could imagine we're replacing the most volatile asset with a negligibly small risk-free rate.)

In comparing the answer here to 1.2, how much risk is your most volatile asset adding to the portfolio? Is this in line with the amount of risk we measured in the stand-alone risk-assessment of `1.1`?

***

# 2. Dynamic Measures

In [67]:
#2.1 
sq_returns= data['portfolio']**2
sigma_t= sq_returns.rolling(window=26).mean().shift(1) **0.5
data['sigma_t']=sigma_t

sigma_end = data['sigma_t'].iloc[-1]
print(sigma_end)

sigma_annualized= sigma_end * (52 ** 0.5)
print(sigma_annualized)

z_05=-1.65
var = z_05 * sigma_end
print("The normal var is", var)

from scipy.stats import norm 
q = 0.05
cvar = -sigma_end * norm.pdf(z_05) / q
print("The normal cvar is", cvar)

# We see that the conditional volatility is lower than the unconditional which tells us that the past 26 weeks 
# have had less volatility than the stock's typical history.

0.03360676359965869
0.2423418187219323
The normal var is -0.055451159939436834
The normal cvar is -0.06873586288717076


## 2.1 

Let's measure the **conditional** statistics of the equally-weighted portfolio of `1.2`, as of the end of the sample.

#### Volatility
For each security, calculate the **rolling** volatility series, $\sigma_t$, with a window of $m=26$.

The value at $\sigma_t$ in the notes denotes the estimate using data through time $t-1$, and thus (potentially) predicting the volatility at $\sigma_{t}$. 

#### Mean
Suppose we can approximate that the daily mean return is zero.

#### VaR
Calculate the **normal VaR** and **normal CVaR** for $q=.05$ and $\tau=1$ as of the end of the sample.Use the approximation, $\texttt{z}_{.05} = -1.65$.

#### Notation Note
In this setup, we are using a forecasted volatility, $\sigma_t$ to estimate the VaR return we would have estimated at the end of $t-1$ in prediction of time $t$.

#### Conclude and Compare
Report
* volatility (annualized).
* normal VaR (.05)
* normal CVaR (.05)

How do these compare to the answers in `1.2`?

In [73]:
#2.2 
sq_returns = data['portfolio']**2

sigma_t_expanding = sq_returns.expanding().mean().shift(1) **0.5
data['sigma_t_expanding']= sigma_t_expanding

z_05 = -1.65
data['var_expanding'] = z_05 * data['sigma_t_expanding']
data['var_rolling']= z_05 * data['sigma_t']

hits_expanding = data['portfolio'] < data['var_expanding']
hits_rolling = data['portfolio'] < data['var_rolling']

hit_rate_expanding = hits_expanding.mean()
hit_rate_rolling = hits_rolling.mean()

print("Hit rate expanding vol is", hit_rate_expanding)
print("Hit rate rolling vol is", hit_rate_rolling)

# These numbers tell us that the hit rates for both are below the targeted 5% which means 
# the models are generally too conservative and predict more risk than what actually happens.
# The rolling vol hit rate being lower than the expanding vol hit rate makes sense since we know that 
# the past 6 months have been more calmer which means that we would get fewer hits.


Hit rate expanding vol is 0.043478260869565216
Hit rate rolling vol is 0.036521739130434785


## 2.2

Backtest the VaR using the **hit test**. Namely, check how many times the realized return at $t$ was smaller than the VaR return calculated using $\sigma_t$, (where again remember the notation in the notes uses $\sigma_t$ as a vol based on data through $t-1$.)

Report the percentage of "hits" using both the 
* expanding volatility
* rolling volatility

***